In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import tensorflow.keras.backend as K
import warnings
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

# ====================================================================
# 1. FUNÇÕES DE CUSTO CUSTOMIZADAS (CUSTOM LOSS FUNCTIONS)
# ====================================================================
def max_sharpe_loss(y_true, y_pred):
    """ Maximiza o Índice de Sharpe da carteira """
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    media_retorno = tf.reduce_mean(retorno_carteira)
    volatilidade = tf.math.reduce_std(retorno_carteira)
    sharpe = media_retorno / (volatilidade + K.epsilon())
    return -sharpe # Retorna negativo porque as redes neurais minimizam o erro

def min_variancia_loss(y_true, y_pred):
    """ Minimiza a Volatilidade / Risco da carteira """
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    variancia = tf.math.reduce_variance(retorno_carteira)
    return variancia

def max_sharpe_loss_institucional(y_true, y_pred):
    """ Maximiza o Sharpe MAS aplica uma multa se a IA concentrar os pesos (HHI) """
    retorno_carteira = tf.reduce_sum(y_true * y_pred, axis=-1)
    sharpe = tf.reduce_mean(retorno_carteira) / (tf.math.reduce_std(retorno_carteira) + K.epsilon())
    
    # Penalização de Concentração (Norma L2 dos pesos)
    penalidade_concentracao = tf.reduce_sum(tf.square(y_pred), axis=-1)
    lambda_reg = 0.05 # Força da multa institucional
    
    return -sharpe + (lambda_reg * tf.reduce_mean(penalidade_concentracao))

# ====================================================================
# 2. ARQUITETURA DA REDE NEURAL "FIM-A-FIM" (END-TO-END)
# ====================================================================
def criar_lstm_otimizadora(look_back, num_features, num_ativos):
    modelo = Sequential()
    # Camadas de extração de padrões temporais
    modelo.add(LSTM(64, return_sequences=True, input_shape=(look_back, num_features)))
    modelo.add(Dropout(0.2))
    modelo.add(LSTM(32))
    
    # A CAMADA MÁGICA: A rede devolve 'num_ativos' pesos. 
    # A ativação Softmax garante que todos os pesos são >= 0 e a soma é exatamente 1.0
    modelo.add(Dense(num_ativos, activation='softmax')) 
    return modelo

# ====================================================================
# 3. CARREGAMENTO DOS DADOS REAIS (SUBSTITUI OS DADOS FICTÍCIOS)
# ====================================================================
print("=== PREPARAÇÃO DOS DADOS REAIS (B3 + NEFIN) ===")
diretorio_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

# 1. Carregar as tuas matrizes reais
df_retornos = pd.read_csv(os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv"), index_col='Data', parse_dates=True)
df_fatores = pd.read_csv(os.path.join(diretorio_dados, "fatores_nefin_limpos.csv"), index_col='Date', parse_dates=True)

# Alinhar os calendários
df_retornos.index = df_retornos.index.tz_localize(None)
df_fatores.index = df_fatores.index.tz_localize(None)
datas_comuns = df_retornos.index.intersection(df_fatores.index)

df_retornos = df_retornos.loc[datas_comuns].fillna(0) # Preenche 0 para ativos que ainda não tinham IPO
df_fatores = df_fatores.loc[datas_comuns]

# Remover o IBOV da matriz de ativos (queremos apenas as ações para a rede comprar)
ativos_para_comprar = [col for col in df_retornos.columns if col != 'IBOV']
df_retornos_ativos = df_retornos[ativos_para_comprar]

num_ativos = len(ativos_para_comprar)
num_features = 5 # Os 5 Fatores Fama-French
look_back = 60

print(f"A preparar tensores para {num_ativos} ativos usando {num_features} fatores...")

# Normalizar os Fatores para a rede neural conseguir ler
scaler_fatores = MinMaxScaler()
fatores_escalados = scaler_fatores.fit_transform(df_fatores.values)

X_real, Y_real = [], []

# Construção do Tensor 3D (X) e Matriz 2D (Y)
for i in range(len(fatores_escalados) - look_back - 1):
    # X_treino: Os 60 dias de passado dos 5 Fatores Macro (O que a rede "vê")
    X_real.append(fatores_escalados[i:(i + look_back), :])
    
    # Y_treino: A fotografia real dos retornos de TODAS as 136 ações no DIA SEGUINTE (O alvo)
    Y_real.append(df_retornos_ativos.values[i + look_back])

X_treino = np.array(X_real)
Y_treino = np.array(Y_real)

print(f"Tensor X_treino pronto: {X_treino.shape} (Dias, Lookback, Features)")
print(f"Tensor Y_treino pronto: {Y_treino.shape} (Dias, Ativos)")

# ====================================================================
# 4. COMPILAÇÃO E TREINO DA IA
# ====================================================================
print(f"\n1. A instanciar a Inteligência Artificial para {num_ativos} ativos...")
modelo_gestor = criar_lstm_otimizadora(look_back, num_features, num_ativos)

# ESCOLHA O SEU GESTOR AQUI:
# Pode mudar a 'loss' para min_variancia_loss ou max_sharpe_loss
print("2. A compilar a Rede Neural com Objetivo de MÁXIMO SHARPE INSTITUCIONAL...")
modelo_gestor.compile(optimizer='adam', loss=max_sharpe_loss_institucional)

print("3. A iniciar o Treino (A IA está a aprender a gerir risco!)...")
historico = modelo_gestor.fit(X_treino, Y_treino, epochs=15, batch_size=64, verbose=1)

# ====================================================================
# 5. TESTE PRÁTICO: A IA GERANDO A CARTEIRA OTIMIZADA
# ====================================================================
print("\n=== RESULTADO: OTIMIZAÇÃO DE PORTFÓLIO OUT-OF-SAMPLE ===")
# Simulamos as informações de mercado do dia de HOJE (Os últimos 60 dias)
X_hoje = np.random.normal(0, 1, size=(1, look_back, num_features))

# Em vez de prever um preço, a rede devolve os PESOS da carteira!
pesos_otimizados = modelo_gestor.predict(X_hoje, verbose=0)[0]

for i, peso in enumerate(pesos_otimizados):
    # Formatação com preenchimento para ficar visualmente alinhado
    print(f"Ativo {i+1:02d}: {peso*100:>5.2f}%")

print("-" * 20)
print(f"Total Investido: {np.sum(pesos_otimizados)*100:>5.1f}%")
print("=======================================================")

=== PREPARAÇÃO DOS DADOS REAIS (B3 + NEFIN) ===
A preparar tensores para 136 ativos usando 5 fatores...
Tensor X_treino pronto: (3902, 60, 5) (Dias, Lookback, Features)
Tensor Y_treino pronto: (3902, 136) (Dias, Ativos)

1. A instanciar a Inteligência Artificial para 136 ativos...
2. A compilar a Rede Neural com Objetivo de MÁXIMO SHARPE INSTITUCIONAL...
3. A iniciar o Treino (A IA está a aprender a gerir risco!)...
Epoch 1/15
61/61 [==============================] - 4s 29ms/step - loss: -0.0489
Epoch 2/15
61/61 [==============================] - 2s 28ms/step - loss: -0.0618
Epoch 3/15
61/61 [==============================] - 2s 28ms/step - loss: -0.0804
Epoch 4/15
61/61 [==============================] - 2s 29ms/step - loss: -0.0879
Epoch 5/15
61/61 [==============================] - 2s 29ms/step - loss: -0.0982
Epoch 6/15
61/61 [==============================] - 2s 29ms/step - loss: -0.0945
Epoch 7/15
61/61 [==============================] - 2s 29ms/step - loss: -0.1001
Epoch 8/15
61